# Train the GRU review summarizer on free Colab compute

This notebook downloads the pinned CC0-tagged Amazon review-title corpus (about 1.29 GiB), creates the deterministic 120,000-row sample, trains the actual TensorFlow GRU encoder–decoder, evaluates the held-out split, verifies artifact reload and autoregressive inference, and exports the artifacts. It does not use a pretrained model or API. Choose **Runtime → Change runtime type → T4 GPU** before running all cells.

In [ ]:
REPO_URL = 'https://github.com/YashBhawarkar/gru-review-summarizer.git'
!git clone --depth 1 $REPO_URL /content/gru-review-summarizer
%cd /content/gru-review-summarizer

In [ ]:
!python -m pip install -q -r requirements-colab.txt
import tensorflow as tf
print('TensorFlow:', tf.__version__, 'GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
!python -m scripts.download_data

In [ ]:
# Change dataset size, epochs, batch size, or seed here. 0 uses all eligible sampled rows.
!python -m scripts.train --dataset-size 0 --epochs 10 --batch-size 128 --seed 42 --patience 2

In [ ]:
!python -m scripts.evaluate --examples 6

In [ ]:
from gru_summarizer.inference import load_summarizer
summarizer = load_summarizer('artifacts')
result = summarizer.summarize('The headphones sound excellent, but the battery dies after three hours and the ear cups are uncomfortable.')
print(result)
assert result.summary is not None

In [ ]:
!pytest -q
!zip -qr /content/gru-review-summarizer-artifacts.zip artifacts
from google.colab import files
files.download('/content/gru-review-summarizer-artifacts.zip')